# 05 - Relational DBMS Storage (SQLite)

This notebook stores the integrated dataset in a SQLite relational database
and runs SQL queries to validate and analyze the data.
- Database: SQLite (flights_weather.db)
- Table: flights
- Queries: 2 analytical SQL queries

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import sqlite3

# Load integrated dataset
integrated_df = pd.read_csv('/content/drive/MyDrive/DM_Project/Data/Integrated/flights_weather_integrated.csv')
integrated_df['FL_DATE'] = pd.to_datetime(integrated_df['FL_DATE'])

print("Integrated dataset loaded:", integrated_df.shape)
print(integrated_df.head())

Integrated dataset loaded: (84356, 35)
     FL_DATE                 AIRLINE                 AIRLINE_DOT AIRLINE_CODE  \
0 2021-06-11    Delta Air Lines Inc.    Delta Air Lines Inc.: DL           DL   
1 2021-06-08  American Airlines Inc.  American Airlines Inc.: AA           AA   
2 2021-06-01   SkyWest Airlines Inc.   SkyWest Airlines Inc.: OO           OO   
3 2021-06-26    Alaska Airlines Inc.    Alaska Airlines Inc.: AS           AS   
4 2022-06-10  Southwest Airlines Co.  Southwest Airlines Co.: WN           WN   

   DOT_CODE  FL_NUMBER ORIGIN            ORIGIN_CITY DEST        DEST_CITY  \
0     19790       2820    ATL            Atlanta, GA  BDL     Hartford, CT   
1     19805       1378    ORD            Chicago, IL  ABQ  Albuquerque, NM   
2     20304       3032    DFW  Dallas/Fort Worth, TX  BIL     Billings, MT   
3     19930       1014    SEA            Seattle, WA  ORD      Chicago, IL   
4     19393       2960    PHX            Phoenix, AZ  LAX  Los Angeles, CA   

   ..

In [3]:
# Create database in Drive
db_path = '/content/drive/MyDrive/DM_Project/Data/flights_weather.db'
conn = sqlite3.connect(db_path)

# Store integrated data
integrated_df.to_sql('flights', conn, if_exists='replace', index=False)

# Verify
count = pd.read_sql_query("SELECT COUNT(*) as total FROM flights", conn)
print("Data stored in SQLite!")
print("Total rows in database:", count['total'][0])

Data stored in SQLite!
Total rows in database: 84356


In [4]:
print("=== QUERY 1: Average Departure Delay per Airport ===")
q1 = pd.read_sql_query("""
    SELECT
        ORIGIN,
        ROUND(AVG(DEP_DELAY), 2) as avg_delay_mins,
        COUNT(*) as total_flights,
        ROUND(AVG(precipitation_mm), 2) as avg_precipitation
    FROM flights
    GROUP BY ORIGIN
    ORDER BY avg_delay_mins DESC
""", conn)
print(q1)

=== QUERY 1: Average Departure Delay per Airport ===
  ORIGIN  avg_delay_mins  total_flights  avg_precipitation
0    MCO           23.10           5359               8.90
1    DEN           19.44          10423               1.97
2    DFW           18.33          11133               3.86
3    ORD           15.23          10375               3.68
4    LAS           14.72           6364               0.13
5    CLT           13.72           7826               2.64
6    ATL           13.72          12790               4.71
7    PHX           13.61           6052               0.07
8    LAX           12.03           7383               0.08
9    SEA            8.37           6651               2.02


In [5]:
print("=== QUERY 2: Flight Delays on Rainy vs Clear Days ===")
q2 = pd.read_sql_query("""
    SELECT
        CASE WHEN precipitation_mm > 0 THEN 'Rainy' ELSE 'Clear' END as weather_type,
        ROUND(AVG(DEP_DELAY), 2) as avg_delay_mins,
        COUNT(*) as total_flights,
        ROUND(AVG(windspeed_max_kmh), 2) as avg_windspeed
    FROM flights
    GROUP BY weather_type
""", conn)
print(q2)

=== QUERY 2: Flight Delays on Rainy vs Clear Days ===
  weather_type  avg_delay_mins  total_flights  avg_windspeed
0        Clear           12.25          42793          20.96
1        Rainy           18.47          41563          18.70


In [6]:
conn.close()
print("SQLite storage complete!")
print("Database saved to Drive: flights_weather.db")

SQLite storage complete!
Database saved to Drive: flights_weather.db
